# EdgeGuard-Road PIDNet-S single-image spike
Execution-only notebook. It does not contain data, credentials, checkpoint bytes, metrics, or scientific acceptance thresholds. It uses only the approved sample contained in the fixed PIDNet checkout and the approved non-commercial academic checkpoint decision.

In [ ]:
EDGEGUARD_REPOSITORY_URL = "https://github.com/OWNER/edgeguard-road.git"
# Replace OWNER with the reviewed repository location.
!git clone {EDGEGUARD_REPOSITORY_URL}
%cd edgeguard-road

In [ ]:
!python -m pip install -e '.[colab,dev]'
!python -m edgeguard doctor --json

## Fixed upstream checkout
This cell checks out only the human-approved official commit under the ignored artifacts tree. It is not vendored or added as a submodule.

In [ ]:
import subprocess
from pathlib import Path

PIDNET_REPOSITORY_URL = "https://github.com/XuJiacong/PIDNet.git"
PIDNET_COMMIT = "4c158cf24ce432f0a8cb43364fae38d93cee0dc3"
PIDNET_CHECKOUT = Path("artifacts/external/pidnet") / PIDNET_COMMIT
PIDNET_CHECKOUT.parent.mkdir(parents=True, exist_ok=True)
if not PIDNET_CHECKOUT.exists():
    subprocess.run(
        ["git", "clone", "--no-checkout", PIDNET_REPOSITORY_URL, str(PIDNET_CHECKOUT)], check=True
    )
subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "checkout", "--detach", PIDNET_COMMIT], check=True
)
actual_commit = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "rev-parse", "HEAD"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert actual_commit == PIDNET_COMMIT, (actual_commit, PIDNET_COMMIT)
actual_origin = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "remote", "get-url", "origin"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
checkout_status = subprocess.run(
    ["git", "-C", str(PIDNET_CHECKOUT), "status", "--porcelain=v1"],
    check=True,
    capture_output=True,
    text=True,
).stdout.strip()
assert actual_origin == PIDNET_REPOSITORY_URL, (actual_origin, PIDNET_REPOSITORY_URL)
assert not checkout_status, checkout_status
print({"repository": actual_origin, "commit": actual_commit, "clean": True})

## Human-controlled checkpoint upload
The old official file link returned HTTP 404 and automatic retrieval failed on 2026-07-26, while the replacement folder linked by the pinned README remained reachable. Obtain `PIDNet_S_Cityscapes_val.pt` only through that official repository-directed folder, then upload exactly that filename. Its checkpoint-specific license remains OPEN QUESTION. Keep it under the ignored artifacts tree, do not redistribute it, and review the printed byte size and SHA-256 before model loading. No alternative direct URL may be invented.

In [ ]:
from google.colab import files

from edgeguard.serialization import sha256_file

CHECKPOINT_SOURCE_URL = (
    "https://drive.google.com/file/d/1JakgBam_GrzyUMp-NbEVVBPEIXLSCssH/view?usp=sharing"
)
CHECKPOINT_COLLECTION_URL = (
    "https://drive.google.com/drive/folders/"
    "0BySIOtxxULinfjlGdGFiT3NQVUdLVDBxWnhhTjB4VXNBRkFOa281WHlkektYY2VBcWVZb1k"
    "?resourcekey=0-w0JIXUekD-FCW-Rm1Z-HfQ&usp=sharing"
)
CHECKPOINT_PATH = Path("artifacts/external/checkpoints/PIDNet_S_Cityscapes_val.pt")
EXPECTED_CHECKPOINT_SHA256 = "REPLACE_WITH_REVIEWED_64_HEX_SHA256"
CHECKPOINT_ACCESS_DATE = "REPLACE_WITH_DOWNLOAD_DATE_YYYY_MM_DD"
SAMPLE_ACCESS_DATE = "REPLACE_WITH_CHECKOUT_ACCESS_DATE_YYYY_MM_DD"

if not CHECKPOINT_PATH.is_file():
    print({"official_repository_file_reference": CHECKPOINT_SOURCE_URL})
    print({"official_replacement_folder": CHECKPOINT_COLLECTION_URL})
    uploaded = files.upload()
    if set(uploaded) != {CHECKPOINT_PATH.name}:
        raise RuntimeError(f"Upload only {CHECKPOINT_PATH.name}; received {sorted(uploaded)}")
    CHECKPOINT_PATH.parent.mkdir(parents=True, exist_ok=True)
    CHECKPOINT_PATH.write_bytes(uploaded[CHECKPOINT_PATH.name])
actual_checkpoint_sha256 = sha256_file(CHECKPOINT_PATH)
print(
    {
        "filename": CHECKPOINT_PATH.name,
        "source_url": CHECKPOINT_SOURCE_URL,
        "official_collection_url": CHECKPOINT_COLLECTION_URL,
        "byte_size": CHECKPOINT_PATH.stat().st_size,
        "sha256": actual_checkpoint_sha256,
        "license_status": "OPEN QUESTION",
    }
)
if EXPECTED_CHECKPOINT_SHA256.startswith("REPLACE_"):
    raise RuntimeError(
        "Review the printed SHA-256, place it in EXPECTED_CHECKPOINT_SHA256, and rerun this cell"
    )
assert actual_checkpoint_sha256 == EXPECTED_CHECKPOINT_SHA256
if CHECKPOINT_ACCESS_DATE.startswith("REPLACE_"):
    raise RuntimeError("Record the actual checkpoint download date")
if SAMPLE_ACCESS_DATE.startswith("REPLACE_"):
    raise RuntimeError("Record the fixed-checkout sample access date")

In [ ]:
command = [
    "python",
    "scripts/run_pidnet_spike.py",
    "--config",
    "configs/pidnet_spike.yaml",
    "--upstream-checkout",
    str(PIDNET_CHECKOUT),
    "--checkpoint",
    str(CHECKPOINT_PATH),
    "--checkpoint-access-date",
    CHECKPOINT_ACCESS_DATE,
    "--expected-checkpoint-sha256",
    EXPECTED_CHECKPOINT_SHA256,
    "--sample-access-date",
    SAMPLE_ACCESS_DATE,
    "--output-dir",
    "artifacts/dev/pidnet_spike",
]
subprocess.run(command, check=True)

In [ ]:
!python -m pytest -q